## Transform Circuits Data (Bronze to Silver)

In the previous step (Bronze), we just loaded raw data from files into a table without any changes. Now we **clean and improve** that data so it's actually useful for analysis.

Think of it like this: Bronze = dumping groceries on the counter. Silver = washing, sorting, and organizing them in the fridge.

**What we do here:**
1. Read the raw data from the Bronze table
2. Remove columns we don't need (like `url`)
3. Rename columns to be clear and consistent (e.g., `lat` becomes `latitude`)
4. Remove rows with missing IDs (bad data)
5. Remove duplicate rows (same circuit listed twice)
6. Fix text formatting (make names look proper - "melbourne" becomes "Melbourne")
7. Save the clean data to the Silver table

#### Loading Configuration
We load shared settings (catalog name, schema names) so we don't have to type them out every time.

In [0]:
%run ../00-common/01.environment-config

In [0]:

bronze_table = f'{catalog_name}.{bronze_schema}.circuit'
silver_table = f'{catalog_name}.{silver_schema}.circuits'

#### Step 1: Read the Bronze Table
We pull the raw circuits data from the Bronze table into a DataFrame. This is our starting point - the "messy" data we need to clean up.

##### How we read it
Instead of reading from a file (like we did in Bronze), we read from a **table** using `spark.read.table()`. This is faster and easier because the data is already stored in Delta format.

In [0]:
circuits_df = spark.read.table(bronze_table)
# or circuits_df = spark.table(bronze_table) best is to use read.table as it is more explicit
display(circuits_df)

#### 2) Keep only the columns required for analytics (DROP url)

In [0]:
from pyspark.sql.functions import *
circuits_selected_df = circuits_df.select(
    col('circuitID'),
    col('circuitName'),
    col('lat'),
    col('long'),
    col('locality'),
    col('country'),
    col('ingestion_timestamp'),
    col('source_file'))

#### Step 2: Drop Unnecessary Columns
The `url` column (Wikipedia links) isn't useful for our analysis, so we remove it. We use `.select()` to pick only the columns we want to keep - everything else gets dropped.

This is like throwing away the packaging and keeping only the food.

#### Rename column

##### Alternative approach (commented out above)
The commented-out code shows `.withColumnRenamed()` (without the S) which renames one column at a time. Both work, but `withColumnsRenamed` is cleaner when renaming multiple columns.

In [0]:
circuits_renamed_df = (circuits_selected_df
 .withColumnsRenamed({
   'circuitID':'circuit_id',                                          
   'circuitName':'circuit_name',                                          
   'lat':'latitude',
   'long':'longitude'})) # this uses withcolumn(S)

#### Step 5: Remove Rows with Missing IDs (Null Check)
If a circuit doesn't have an ID (`circuit_id` is NULL), it's useless data - we can't identify or join it with anything else. So we **filter it out**.

This is called a **business key validation** - the `circuit_id` is the key that identifies each circuit, so it MUST have a value.

We use `.filter(col('circuit_id').isNotNull())` which keeps only rows where the ID exists.

In [0]:
display(circuits_renamed_df)


In [0]:
circuits_validate_df = circuits_renamed_df.filter(
    col('circuit_id').isNotNull() 
)

In [0]:
display(circuits_validate_df)  

#### Step 6: Remove Duplicates
Sometimes the same circuit appears more than once (maybe it was in multiple files, or loaded twice). We remove these duplicates so each circuit appears exactly **once**.

Two ways to do this:
- `.distinct()` - removes rows that are 100% identical across ALL columns
- `.dropDuplicates(["circuit_id"])` - removes rows with the same `circuit_id`, keeping the first one found

We use `dropDuplicates` because two rows might have slightly different values but the same ID - we only want one row per circuit.

In [0]:
circuits_distinct_df = circuits_validate_df.distinct()

In [0]:
circuits_distinct_df = circuits_validate_df.dropDuplicates(["circuit_id"])
display(circuits_distinct_df)

In [0]:
circuits_distinct_df.createOrReplaceTempView("circuits_distinct_df")
spark.sql("select count(*) from circuits_distinct_df").display()

#### Step 7: Fix Text Formatting (Title Case)
Some text values are all lowercase ("melbourne") or inconsistent. We convert them to **Title Case** ("Melbourne") so they look proper and consistent.

We use `initcap()` which capitalizes the first letter of each word.

**Why `.withColumn()` here?** Unlike renaming, we're *changing the values* inside a column. `.withColumn()` either creates a new column or overwrites an existing one with transformed values. Since we use the same name (`circuit_name`), it overwrites the old messy values with the clean Title Case versions.

In [0]:
from pyspark.sql.functions import initcap
circuits_final_df =(circuits_distinct_df
 .withColumn('circuit_name', initcap(col('circuit_name')))
 .withColumn('locality', initcap(col('locality'))))
display(circuits_final_df)



#### Step 8: Write to Silver Table
Finally, we save our clean, transformed data to the Silver layer. This table is now ready for analysis - no more messy names, no duplicates, no nulls.

Anyone querying the Silver table gets reliable, consistent data without having to clean it themselves.

In [0]:
(
  circuits_final_df
  .write
  .mode("overwrite")
  .format("delta")
  .saveAsTable(f'{catalog_name}.{silver_schema}.circuits')
)

In [0]:
spark.read.table(silver_schema).display()